# Análisis de slices con métricas agregadas y bootstrap

Este notebook muestra una forma rápida de analizar consultas por slices, calcular métricas y resumirlas con intervalo de confianza bootstrap.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(42)
print('Bibliotecas listas')

Bibliotecas listas


## 2. Cargar Datos

A continuación se crea un conjunto pequeño de consultas con una métrica de slice, suficiente para ver cómo se agregan los resultados.

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Datos sintéticos pequeños para demo rápida
rows = [
    {"query": "¿Cómo registrar una marcación?", "slice": "marcaciones", "score": 0.91},
    {"query": "¿Cómo corregir una tardanza?", "slice": "marcaciones", "score": 0.88},
    {"query": "¿Cómo ver permisos de vacaciones?", "slice": "permisos", "score": 0.72},
    {"query": "¿Cómo solicitar un permiso?", "slice": "permisos", "score": 0.69},
    {"query": "¿Cómo revisar horas extras?", "slice": "horas_extras", "score": 0.84},
    {"query": "¿Cómo reportar horas extras?", "slice": "horas_extras", "score": 0.80},
    {"query": "¿Cómo cambiar mi contraseña?", "slice": "configuracion", "score": 0.65},
    {"query": "¿Cómo activar MFA?", "slice": "configuracion", "score": 0.60},
]

df = pd.DataFrame(rows)
df.head()

,query,slice,score
0,¿Cómo registrar una marcación?,marcaciones,0.91
1,¿Cómo corregir una tardanza?,marcaciones,0.88
2,¿Cómo ver permisos de vacaciones?,permisos,0.72
3,¿Cómo solicitar un permiso?,permisos,0.69
4,¿Cómo revisar horas extras?,horas_extras,0.84


In [5]:
## 3. Calcular métricas por slice

import pandas as pd
import numpy as np

np.random.seed(42)

rows = [
    {"query": "¿Cómo registrar una marcación?", "slice": "marcaciones", "score": 0.91},
    {"query": "¿Cómo corregir una tardanza?", "slice": "marcaciones", "score": 0.88},
    {"query": "¿Cómo ver permisos de vacaciones?", "slice": "permisos", "score": 0.72},
    {"query": "¿Cómo solicitar un permiso?", "slice": "permisos", "score": 0.69},
    {"query": "¿Cómo revisar horas extras?", "slice": "horas_extras", "score": 0.84},
    {"query": "¿Cómo reportar horas extras?", "slice": "horas_extras", "score": 0.80},
    {"query": "¿Cómo cambiar mi contraseña?", "slice": "configuracion", "score": 0.65},
    {"query": "¿Cómo activar MFA?", "slice": "configuracion", "score": 0.60},
]

df = pd.DataFrame(rows)

summary = df.groupby('slice')['score'].agg(['mean', 'count'])
summary = summary.rename(columns={'mean': 'score_mean', 'count': 'n'})
summary

,score_mean,n
slice,,
configuracion,0.625,2
horas_extras,0.820,2
marcaciones,0.895,2
permisos,0.705,2


In [7]:
## 3. Calcular métricas por slice

import pandas as pd
import numpy as np

np.random.seed(42)

rows = [
    {"query": "¿Cómo registrar una marcación?", "slice": "marcaciones", "score": 0.91},
    {"query": "¿Cómo corregir una tardanza?", "slice": "marcaciones", "score": 0.88},
    {"query": "¿Cómo ver permisos de vacaciones?", "slice": "permisos", "score": 0.72},
    {"query": "¿Cómo solicitar un permiso?", "slice": "permisos", "score": 0.69},
    {"query": "¿Cómo revisar horas extras?", "slice": "horas_extras", "score": 0.84},
    {"query": "¿Cómo reportar horas extras?", "slice": "horas_extras", "score": 0.80},
    {"query": "¿Cómo cambiar mi contraseña?", "slice": "configuracion", "score": 0.65},
    {"query": "¿Cómo activar MFA?", "slice": "configuracion", "score": 0.60},
]

df = pd.DataFrame(rows)

summary = df.groupby('slice')['score'].agg(['mean', 'count'])
summary = summary.rename(columns={'mean': 'score_mean', 'count': 'n'})
summary

,score_mean,n
slice,,
configuracion,0.625,2
horas_extras,0.820,2
marcaciones,0.895,2
permisos,0.705,2


## 4. Agregar por slice con IC bootstrap

Aquí se calcula la media por slice, el intervalo de confianza bootstrap y se marca qué slices son problemáticos.


In [8]:
def bootstrap_ci(vals, n_boot=500, alpha=0.05):
    vals = np.asarray(vals, dtype=float)
    if len(vals) < 2:
        return (float(vals[0]), float(vals[0]))
    boot = [np.mean(np.random.choice(vals, len(vals), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(boot, 100 * alpha / 2)
    hi = np.percentile(boot, 100 * (1 - alpha / 2))
    return (float(lo), float(hi))

summary = df.groupby('slice')['score'].agg(['mean', 'count'])
summary = summary.rename(columns={'mean': 'score_mean', 'count': 'n'}).reset_index()

rows = []
for _, row in summary.iterrows():
    vals = df.loc[df['slice'] == row['slice'], 'score'].tolist()
    lo, hi = bootstrap_ci(vals)
    rows.append({
        'slice': row['slice'],
        'n': row['n'],
        'score_mean': round(row['score_mean'], 3),
        'ic_low': round(lo, 3),
        'ic_high': round(hi, 3),
        'estado': 'problematico' if row['score_mean'] < 0.75 else 'ok'
    })

result = pd.DataFrame(rows)
result

,slice,n,score_mean,ic_low,ic_high,estado
0,configuracion,2,0.625,0.60,0.65,problematico
1,horas_extras,2,0.820,0.80,0.84,ok
2,marcaciones,2,0.895,0.88,0.91,ok
3,permisos,2,0.705,0.69,0.72,problematico


In [9]:
## 5. Visualizar resultados de slices

import matplotlib.pyplot as plt

result.style.background_gradient(subset=['score_mean'], cmap='RdYlGn')

,slice,n,score_mean,ic_low,ic_high,estado
0,configuracion,2,0.625000,0.600000,0.650000,problematico
1,horas_extras,2,0.820000,0.800000,0.840000,ok
2,marcaciones,2,0.895000,0.880000,0.910000,ok
3,permisos,2,0.705000,0.690000,0.720000,problematico


In [10]:
## 5. Visualizar resultados de slices

import matplotlib.pyplot as plt

result.style.background_gradient(subset=['score_mean'], cmap='RdYlGn')

,slice,n,score_mean,ic_low,ic_high,estado
0,configuracion,2,0.625000,0.600000,0.650000,problematico
1,horas_extras,2,0.820000,0.800000,0.840000,ok
2,marcaciones,2,0.895000,0.880000,0.910000,ok
3,permisos,2,0.705000,0.690000,0.720000,problematico


In [11]:
## 6. Mitigación para slices problemáticos

import pandas as pd

mitigation_plan = []
for _, row in result.iterrows():
    if row['estado'] != 'problematico':
        continue
    if row['slice'] == 'permisos':
        accion = 'Ampliar la documentación de permisos, añadir sinónimos y ajustar el retriever.'
    elif row['slice'] == 'configuracion':
        accion = 'Separar la categoría en subtemas y crear chunks más específicos.'
    else:
        accion = 'Recolectar más ejemplos y revisar el vocabulario de las consultas.'

    mitigation_plan.append({
        'slice': row['slice'],
        'score_mean': row['score_mean'],
        'prioridad': 'Alta' if row['score_mean'] < 0.7 else 'Media',
        'mitigacion': accion
    })

mitigation_df = pd.DataFrame(mitigation_plan)
mitigation_df

,slice,score_mean,prioridad,mitigacion
0,configuracion,0.625,Alta,Separar la categoría en subtemas y crear chunk...
1,permisos,0.705,Media,"Ampliar la documentación de permisos, añadir s..."
